# 07. Measurements and tables

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner

## What you will learn
- Measure labeled objects
- Create pandas tables
- Interpret area, centroid, shape, and intensity
- Convert pixels to physical units carefully

> **Learning rule:** understand the problem first, then choose the function.

## 1. Measure only after segmentation QC

Measurements are properties of the segmentation. If the mask is wrong, the table is wrong even if the code executes perfectly.

In [ ]:
import pandas as pd
import skimage as ski
from skimage import filters, measure, morphology
image = ski.data.coins()
mask = filters.gaussian(image,1) > filters.threshold_otsu(filters.gaussian(image,1))
mask = morphology.remove_small_objects(mask, max_size=80)
labels = measure.label(mask)

## 2. Build a measurement table

`regionprops_table()` is convenient when you want one object per row and columns suitable for pandas/CSV workflows.

In [ ]:
# intensity_image is required for intensity properties such as mean_intensity.
props = measure.regionprops_table(
    labels, intensity_image=image,
    properties=("label","area","centroid","eccentricity","solidity","mean_intensity")
)
df = pd.DataFrame(props)
print(df.head())
print("objects:", len(df))

## 3. Pixels are not micrometers

Convert to physical units only with acquisition metadata or validated calibration.

In [ ]:
# Example calibration ONLY: replace with real metadata for real experiments.
pixel_size_y_um = 0.5
pixel_size_x_um = 0.5
df["area_um2"] = df["area"] * pixel_size_y_um * pixel_size_x_um
# Save a flat, shareable results table.
df.to_csv("../outputs/object_measurements.csv", index=False)
print(df[["label","area","area_um2","mean_intensity"]].head())

## Function-selection guide

| Goal | Function | Note |
|---|---|---|
| One-object-per-row table | `regionprops_table()` | Best for pandas/CSV |
| Custom per-object logic | `regionprops()` | Returns region objects |
| Data table | `pd.DataFrame()` | Organize measurements |
| Export | `df.to_csv()` | Save derived results |

**Units:** record pixel size and axis spacing used for every physical conversion.

## Takeaway

**Choose functions because they solve a specific image problem, and always inspect the result before measuring.**